# Indexes: Making Queries Faster

When you filter or join on a column, the database may have to scan every row
to find matches. On small tables this is fine, but on large tables it is
slow. An **index** is a data structure (usually a B-tree) that lets the
database jump straight to matching rows, like the index at the back of a
book. This notebook shows how indexes work, how to see their effect and how
to add them to your Library in pgModeler.

## Seeing the problem: EXPLAIN ANALYZE

PostgreSQL can show you how it runs a query, and how long it takes, with
**EXPLAIN ANALYZE**. Let's look at a query on Chinook's track table, finding
all tracks composed by Steve Harris.

In [ ]:
from rel_db_learning import get_chinook_connection, get_project_connection, run_sql, run_ddl

chinook = get_chinook_connection()
project = get_project_connection()

run_sql("EXPLAIN ANALYZE SELECT * FROM track WHERE composer = 'Steve Harris'", chinook)

Read the plan from the bottom up. The key line is **Seq Scan on track**:
PostgreSQL scanned the entire table, row by row. Notice **Rows Removed by
Filter**: it checked thousands of rows and threw most away to find the few
that match. On a large table, this work adds up.

## Adding an index

An index on the composer column lets PostgreSQL find matching rows without
scanning everything. The SQL to create one is:

```sql
CREATE INDEX index_name ON table (column);
```

On Chinook, just to see the effect, we create the index directly. (In your
own project you will use pgModeler, shown later.) We use `IF NOT EXISTS` so
the cell can be run more than once.

In [ ]:
run_ddl("CREATE INDEX IF NOT EXISTS idx_track_composer ON track (composer)", chinook)

Now run the same EXPLAIN ANALYZE again and compare:

In [ ]:
run_sql("EXPLAIN ANALYZE SELECT * FROM track WHERE composer = 'Steve Harris'", chinook)

The plan should now mention an **Index Scan** or **Bitmap Index Scan** on
`idx_track_composer` instead of a full Seq Scan. PostgreSQL uses the index to
locate the matching rows directly.

Before: Seq Scan, Execution Time 5.153 ms, scanned 3503 lines.
After: Bitmap Index Scan in idx_track_composer, Execution Time 0.089 ms, aprox 58 times faster.
Heap Blocks: exact=5 it read only 5 blocks instead of the whole table). 

Note: PostgreSQL is cost-based, so on small tables it sometimes still chooses a Seq Scan because scanning a small table is cheap. The benefit of an index grows with table size. On a table with millions of rows, the difference between a Seq Scan and an Index Scan can be enormous.

If you want to remove the demonstration index from Chinook, you can drop it:

In [ ]:
# Optional cleanup:
#run_ddl("DROP INDEX IF EXISTS idx_track_composer", chinook)

## Indexes in your own project, via pgModeler

Indexes are schema objects, so in your Library you define them in pgModeler,
the same round-trip you use for tables and columns. Suppose you often search
books by title. An index on `book.title` would speed that up.

### Step 1: Add the index in pgModeler

1. Open your `stage_4_design.dbm` file in pgModeler. (Indexes do not start a
   new stage, so you can keep editing the Stage 4 design.)
2. Double-click the **book** table and open the **Indexes** tab.
3. Add a new index:
   - Give it a name, for example `idx_book_title`.
   - Add an element referring to the `title` column.
4. Save the `.dbm` file.

### Step 2: Generate the change with Diff and apply it

1. Open pgModeler's Diff tool, source = your model, target = your my_project
   database (the connection from notebook 08).
2. Generate the diff. It should contain a single CREATE INDEX statement for
   `idx_book_title`.
3. Review it, copy it, paste it below, and run it.

In [ ]:
# Paste the CREATE INDEX statement that pgModeler's Diff generated.
index_sql = """
-- Paste your pgModeler Diff output here, for example:
-- CREATE INDEX idx_book_title ON public.book USING btree (title);
"""

run_ddl(index_sql, project)

Let's confirm the index now exists on your book table:

In [ ]:
run_sql("""
    SELECT indexname, indexdef
    FROM pg_indexes
    WHERE tablename = 'book'
""", project)

Your Library is small, so PostgreSQL may not need this index yet, but it is
there for when your data grows. You can inspect how a query would run with
EXPLAIN ANALYZE on your own tables too:

In [ ]:
run_sql("EXPLAIN ANALYZE SELECT * FROM book WHERE title = 'The Hobbit'", project)

## When to add an index

Indexes are not free. They speed up reads but slow down writes (every INSERT,
UPDATE, or DELETE must also update the index), and they take storage. Good
candidates for an index:

- Columns you frequently filter on in WHERE clauses
- Columns you join on (foreign keys are common choices)
- Columns you often sort by with ORDER BY

Poor candidates:

- Columns that are rarely queried
- Very small tables, where a scan is already fast
- Columns with few distinct values (for example a boolean), where an index
  helps little

A practical rule: add indexes to match how your application actually queries
the data, and measure with EXPLAIN ANALYZE rather than guessing.

## Experiment freely

Try EXPLAIN ANALYZE on other Chinook queries, with and without an index. For
example, create an index on `track.milliseconds` and compare a range query
before and after.

In [ ]:
run_sql("EXPLAIN ANALYZE SELECT name FROM track WHERE milliseconds > 400000", chinook)

## What's next

You can now make queries faster with indexes and read query plans. Next you
will learn about **domains**, custom data types with built-in validation,
and you will add validated contact information to your members.